In [4]:
import pandas as pd
import requests
from bs4 import BeautifulSoup as bs

In [47]:
#Need to request 
# https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX_History.csv
# When I paste that link into my search bar, it downloads a csv file
# I just want to be able to read it into a pandas dataframe 
# without having to download it first. 
# I think I can use requests to do that

url = "https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX_History.csv"

def get_cboe_data(url):
    response = requests.get(url)
    if response.status_code == 200:
        # Read the CSV data into a pandas DataFrame
        import io
        vix_data = pd.read_csv(io.StringIO(response.text))
        vix_data['DATE'] = pd.to_datetime(vix_data['DATE'])
        return vix_data
    else:
        print(f"Failed to retrieve data. Status code: {response.status_code}")
        return None
    
#
#MASSIVE NOTE FOR AGENT: SOMETIMES THE REQUEST SILENTLY FAILS - THE FAILURE IS THE FOLLOWING:
#I GET THE DATA RETURNED, NO ERROR, BUT THE DATA IS NOT UP TO DATE
#IN THE PROD ETL PIPELINE, IF DATE IS MORE THAN 1 DAY OLD, LOG A WARNING AND TRY AGAIN.
#AFTER 3 FAILURES, RAISE AN ERROR AND STOP THE PIPELINE.

In [53]:
#Examine data
print(get_cboe_data(url).head())

vix_data = get_cboe_data(url)
print(vix_data.max())

        DATE   OPEN   HIGH    LOW  CLOSE
0 1990-01-02  17.24  17.24  17.24  17.24
1 1990-01-03  18.19  18.19  18.19  18.19
2 1990-01-04  19.22  19.22  19.22  19.22
3 1990-01-05  20.11  20.11  20.11  20.11
4 1990-01-08  20.26  20.26  20.26  20.26
DATE     2026-07-16 00:00:00
OPEN                   82.69
HIGH                   89.53
LOW                    72.76
CLOSE                  82.69
dtype: object


In [48]:
#Next URL to try
#https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX3M_History.csv

ViX_3M = get_cboe_data(url="https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX3M_History.csv")


In [50]:
print(ViX_3M.head())

        DATE   OPEN   HIGH    LOW  CLOSE
0 2009-09-18  25.91  26.66  25.91  26.54
1 2009-09-21  26.96  27.24  26.17  26.23
2 2009-09-22  25.97  26.21  25.65  25.69
3 2009-09-23  25.70  26.20  25.01  26.09
4 2009-09-24  26.10  27.42  25.99  27.13


In [52]:
#print(ViX_3M.info())
print(ViX_3M.describe())
print(ViX_3M['DATE'].min(), ViX_3M['DATE'].max())

                             DATE         OPEN         HIGH          LOW  \
count                        4232  4232.000000  4232.000000  4232.000000   
mean   2018-02-13 16:13:09.413988    20.481425    21.123733    19.910324   
min           2009-09-18 00:00:00    11.870000    11.910000    10.060000   
25%           2013-12-01 06:00:00    16.000000    16.350000    15.650000   
50%           2018-02-13 12:00:00    19.230000    19.790000    18.750000   
75%           2022-04-27 06:00:00    23.322500    24.032500    22.480000   
max           2026-07-17 00:00:00    73.140000    86.610000    65.150000   
std                           NaN     6.019431     6.499802     5.675623   

             CLOSE  
count  4232.000000  
mean     20.443459  
min      11.850000  
25%      15.990000  
50%      19.245000  
75%      23.200000  
max      72.980000  
std       6.032184  
2009-09-18 00:00:00 2026-07-17 00:00:00


In [54]:
#Next URL to try
# https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX9D_History.csv

ViX_9D = get_cboe_data(url="https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX9D_History.csv")

In [55]:
print(ViX_9D.head())
print(ViX_9D["DATE"].min(), ViX_9D["DATE"].max())

        DATE   OPEN   HIGH    LOW  CLOSE
0 2011-01-04  16.06  16.06  16.06  16.06
1 2011-01-05  15.57  15.57  15.57  15.57
2 2011-01-06  15.71  15.71  15.71  15.71
3 2011-01-07  15.01  15.01  15.01  15.01
4 2011-01-10  15.81  15.81  15.81  15.81
2011-01-04 00:00:00 2026-07-17 00:00:00


In [ ]:
# | VXX/UVXY split-adjusted daily prices | yfinance (interim) → validate against a second source | Free | VXX 2009 → present |
# Need to get VXX/UVXY split-adjusted daily prices from yfinance. 

import yfinance as yf

vxx = yf.Ticker("VXX").history(period="max", auto_adjust=True)
uvxy = yf.Ticker("UVXY").history(period="max", auto_adjust=True)

print(vxx.index.min(), vxx.index.max())
print(uvxy.index.min(), uvxy.index.max())
print(vxx.head())

#NOTE FOR AGENT: ALWAYS HAVE AUTO_ADJUST = True AND THERE IS NO Adj_Close column, WITH AUTO_ADJUST = TRUE, THE CLOSE COLUMN IS ALREADY ADJUSTED FOR SPLITS AND DIVIDENDS. DO NOT USE THE Adj_Close COLUMN.
#Date min and date max is 2018-01-25 -> 2026-07-17 for vxx
#Date min and date max is 2011-10-04 -> 2026-07-17 for uvxy



2018-01-25 00:00:00-05:00 2026-07-17 00:00:00-04:00
2011-10-04 00:00:00-04:00 2026-07-17 00:00:00-04:00
                                  Open         High          Low        Close  \
Date                                                                            
2018-01-25 00:00:00-05:00  1770.239990  1770.239990  1770.239990  1770.239990   
2018-01-26 00:00:00-05:00  1770.239990  1770.239990  1770.239990  1770.239990   
2018-01-29 00:00:00-05:00  1868.800049  1893.119995  1868.800049  1893.119995   
2018-01-30 00:00:00-05:00  1970.560059  2035.839966  1951.359985  1955.199951   
2018-01-31 00:00:00-05:00  1917.439941  1963.520020  1917.439941  1961.599976   

                           Volume  Dividends  Stock Splits  Capital Gains  
Date                                                                       
2018-01-25 00:00:00-05:00       0        0.0           0.0            0.0  
2018-01-26 00:00:00-05:00       0        0.0           0.0            0.0  
2018-01-29 00:00:00-05:0